# 05 Final Load Prep

Use this notebook to compute final KPIs, prepare the Tableau-ready dataset, and export the exact file used in the dashboard.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

In [2]:
DATA_PATH = 'https://raw.githubusercontent.com/pushkar-bit/Section-E_G-19_Healthcare-Patient-Readmission-Intelligence/main/data/processed/cleaned_hospital_readmissions.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'

df = pd.read_csv(DATA_PATH)
df.head()

,age,time_in_hospital,n_lab_procedures,n_procedures,n_medications,n_outpatient,n_inpatient,n_emergency,medical_specialty,diag_1,...,total_visits,is_frequent_patient,stay_intensity,procedures_per_day,meds_per_visit,service_utilization_score,length_of_stay_bucket,age_numeric,age_group,is_senior
0,3,8,72,1,18,2,0,0,4,0,...,2,0,576,0,9,34.5,0,75.0,3,1
1,3,3,34,2,13,0,0,0,5,6,...,0,0,102,1,13,18.1,2,75.0,3,1
2,1,5,45,0,18,0,0,0,4,0,...,0,0,225,0,18,23.4,1,55.0,2,0
3,3,2,36,0,12,1,0,0,4,0,...,1,0,72,0,12,18.0,2,75.0,3,1
4,2,1,42,0,7,0,0,0,3,6,...,0,0,42,0,7,18.9,2,65.0,3,1


#Core KPIs

In [3]:
kpis = {
    "Total Patients": len(df),
    "Readmission Rate (%)": round(df["readmitted_30_days"].mean() * 100, 2),
    "Avg Length of Stay": round(df["time_in_hospital"].mean(), 2),
    "Avg Medications": round(df["n_medications"].mean(), 2),
    "Avg Prior Visits": round(df["total_visits"].mean(), 2),
}

kpi_df = pd.DataFrame(list(kpis.items()), columns=["Metric", "Value"])
display(kpi_df)

,Metric,Value
0,Total Patients,25000.00
1,Readmission Rate (%),47.02
2,Avg Length of Stay,4.45
3,Avg Medications,16.25
4,Avg Prior Visits,1.17



##High-Risk Patient Segment

In [4]:
# Define high-risk patients (simple rule based on your EDA)
df["high_risk_flag"] = (
    (df["n_inpatient"] > 2) |
    (df["total_visits"] > 3)
).astype(int)

risk_summary = df["high_risk_flag"].value_counts().rename({
    0: "Low Risk",
    1: "High Risk"
})

display(risk_summary)

print(f"High-risk patients: {round(risk_summary[1] / len(df) * 100, 2)}%")

,count
high_risk_flag,
Low Risk,22154
High Risk,2846


High-risk patients: 11.38%


/tmp/ipykernel_7248/2302981787.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"High-risk patients: {round(risk_summary[1] / len(df) * 100, 2)}%")


##KPI by Segment

In [5]:
# Create high-risk flag
df["high_risk_flag"] = (
    (df["n_inpatient"] > 2) |
    (df["total_visits"] > 3)
).astype(int)
segment_kpis = (
    df.groupby("high_risk_flag")["readmitted_30_days"]
    .mean() * 100
).round(2)

segment_kpis.index = ["Low Risk", "High Risk"]
display(segment_kpis)

print(
    f"Insight: High-risk patients have {segment_kpis[1]}% readmission vs {segment_kpis[0]}% for low-risk."
)

,readmitted_30_days
Low Risk,43.93
High Risk,71.05


Insight: High-risk patients have 71.05% readmission vs 43.93% for low-risk.


/tmp/ipykernel_7248/3862143241.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  f"Insight: High-risk patients have {segment_kpis[1]}% readmission vs {segment_kpis[0]}% for low-risk."


##Operational KPI

In [6]:
ops_kpi = (
    df.groupby("medical_specialty")["readmitted_30_days"]
    .mean() * 100
).round(2).sort_values(ascending=False)

display(ops_kpi.head(5))

print(
    f"Insight: Highest readmission observed in {ops_kpi.index[0]} "
    f"({ops_kpi.iloc[0]}%) → operational focus area."
)

,readmitted_30_days
medical_specialty,
2,49.52
1,49.39
4,48.91
0,45.00
3,44.77


Insight: Highest readmission observed in 2 (49.52%) → operational focus area.


#Final Dataset for Use

In [7]:
final_df = df[[
    "readmitted_30_days",
    "high_risk_flag",
    "n_inpatient",
    "total_visits",
    "time_in_hospital",
    "n_medications",
    "medical_specialty"
]]

display(final_df.head())

,readmitted_30_days,high_risk_flag,n_inpatient,total_visits,time_in_hospital,n_medications,medical_specialty
0,0,0,0,2,8,18,4
1,0,0,0,0,3,13,5
2,1,0,0,0,5,18,4
3,1,0,0,1,2,12,4
4,0,0,0,0,1,7,3


In [8]:
final_df.to_csv("final_readmission_dataset.csv", index=False)

print("Final dataset saved successfully.")

Final dataset saved successfully.


In [9]:
TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(TABLEAU_READY_PATH, index=False)
print(f'Saved Tableau-ready dataset to {TABLEAU_READY_PATH}')

Saved Tableau-ready dataset to /content/data/processed/tableau_ready_dataset.csv
